In [1]:
import faiss
import numpy as np
import pandas as pd

In [2]:
embeddings = np.load(
    "../data/vector_store/event_embeddings.npy"
)

documents = pd.read_parquet(
    "../data/vector_store/event_documents.parquet"
)

index = faiss.read_index(
    "../data/vector_store/events.faiss"
)

In [3]:
print("Embeddings :", embeddings.shape)
print("Documents :", documents.shape)
print("FAISS :", index.ntotal)

Embeddings : (14134, 1024)
Documents : (14134, 17)
FAISS : 14134


In [4]:
len(embeddings) == len(documents) == index.ntotal

True

## Le test de recherche 

In [5]:
query = "Je cherche un spectacle pour enfants"

In [6]:
import os
from mistralai.client import Mistral

client = Mistral(
    api_key=os.getenv("MISTRAL_KEY")
)

response = client.embeddings.create(
    model="mistral-embed",
    inputs=[query],
)

In [8]:
query_embedding = np.array(
    [response.data[0].embedding],
    dtype=np.float32,
)

In [18]:
faiss.normalize_L2(query_embedding)

In [19]:
scores, indices = index.search(
    query_embedding,
    k=5,
)

In [20]:
indices

array([[10313,  2522, 10763, 10597, 10604]])

In [21]:
scores

array([[0.8083968, 0.7966311, 0.7948594, 0.7948582, 0.7922871]],
      dtype=float32)

In [22]:
results = documents.iloc[indices[0]].copy()
results["score"] = scores[0]

display(
    results
)

,uid,title,chunk_id,chunk_text,description,longDescription,dateRange,firstTiming,lastTiming,location_name,location_address,location_postal_code,location_latitude,location_longitude,image,registration,status,score
10313,42667655,Hulul,42667655_1,"jour Pour profiter pleinement, découvrez le gu...",Jeune public / Musique & marionnettes / Cie Ch...,Hulul est un hibou solitaire qui se pose des q...,"Samedi 21 novembre, 11h00",Du 21/11/2026 à 11h00 au 21/11/2026 à 11h40,Du 21/11/2026 à 11h00 au 21/11/2026 à 11h40,Centre culturel Renan,5 chemin d'Audibert 31200 Toulouse,31200,43.636404,1.442497,https://img.openagenda.com/main/6f5ed1559bb841...,NaN,Programmé,0.808397
2522,55020577,Faubourg,55020577_1,rez le guide illustré du très jeune spectateur...,Spectacle jeune public dès 4 ans - Marionnette...,Une maquette d’immeubles construite comme un q...,"Samedi 22 novembre 2025, 11h00",Du 22/11/2025 à 11h00 au 22/11/2025 à 11h40,Du 22/11/2025 à 11h00 au 22/11/2025 à 11h40,Centre culturel Renan,5 chemin d'Audibert 31200 Toulouse,31200,43.636404,1.442497,https://cdn.openagenda.com/main/b908c754005742...,NaN,Programmé,0.796631
10763,51552904,Ma forêt imaginaire,51552904_1,"t, découvrez le guide illustré du très jeune s...",Danse / De 6 mois à 6 ans / Cie Lamdanse,Ma forêt imaginaire est un voyage sensoriel et...,"Dimanche 1 février, 10h30",Du 01/02/2026 à 10h30 au 01/02/2026 à 11h00,Du 01/02/2026 à 10h30 au 01/02/2026 à 11h00,Centre culturel Alban-Minville,"1 Place Martin Luther King, 31100 Toulouse, Fr...",31100,43.564568,1.399079,https://img.openagenda.com/main/615e0054909347...,NaN,Programmé,0.794859
10597,73661787,Granita et le bazar des émotions,73661787_1,ectateur. Vous n'y trouverez aucune leçon mais...,"Jeune public / Théâtre, cirque & musique / Cie...",Granita est une petite fille italienne de 7 an...,"Mercredi 22 avril, 14h30",Du 22/04/2026 à 14h30 au 22/04/2026 à 15h00,Du 22/04/2026 à 14h30 au 22/04/2026 à 15h00,Centre culturel Saint-Simon,"10 Chemin de Liffard, 31100 Toulouse, France",31100,43.561769,1.379015,https://img.openagenda.com/main/114ede08062a4c...,NaN,Programmé,0.794858
10604,79490491,Petits Mondes Lumineux,79490491_1,ouverez aucune leçon mais quelques clés et con...,Jeune public / Arts croisés & expérience senso...,_Petits mondes lumineux_ est une installation ...,"Samedi 18 avril, 10h30",Du 18/04/2026 à 10h30 au 18/04/2026 à 11h15,Du 18/04/2026 à 10h30 au 18/04/2026 à 11h15,Centre culturel Bonnefoy,4 Rue Du Faubourg Bonnefoy 31500 toulouse,31500,43.616310,1.453587,https://img.openagenda.com/main/6b6ffca37e8f4e...,NaN,Programmé,0.792287


In [23]:
test_queries = [
    "Je cherche une exposition de peinture",
    "Je cherche un activité pour enfants",
    "Je veux écouter du jazz",
    "Je cherche une pièce de théâtre",
    "Que faire en famille avec de jeunes enfants ?",
    "Je voudrais une activité culturelle en plein air",
    "Je cherche quelque chose autour de la photographie",
    "Je cherche une activité gratuite",
    "Je voudrais une activité accessible aux personnes à mobilité réduite",
    "Je cherche un événement pour adolescents",
]

for query in test_queries:

    # Embedding de la requête
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[query],
    )

    query_embedding = np.array(
        [response.data[0].embedding],
        dtype=np.float32,
    )

    # Normalisation pour la similarité cosinus
    faiss.normalize_L2(query_embedding)

    # Recherche des 5 résultats les plus proches
    scores, indices = index.search(
        query_embedding,
        k=5,
    )

    # Récupération des événements
    results = documents.iloc[indices[0]].copy()
    results["score"] = scores[0]

    print("\n" + "=" * 100)
    print(f"REQUÊTE : {query}")
    print("=" * 100)

    display(
        results[
            [
                "title",
                "description",
                "longDescription",
                "dateRange",
                "location_name",
                "score",
            ]
        ]
    )


REQUÊTE : Je cherche une exposition de peinture


,title,description,longDescription,dateRange,location_name,score
12793,[La galerie 3.1] Nous ne sommes pas séparés,"Avec Marta Anglada, Christophe Debens, Richard...",**Organisée en partenariat avec** [**Les Arts ...,18 septembre 2025 - 10 janvier 2026,La galerie 3.1 - Conseil départemental de la H...,0.808023
13869,"""La traversée des apparences""",Exposition de peinture post-impréssionniste,La Communauté Municipale de Santé est heureuse...,13 juillet - 28 août,Communauté Municipale de Santé - CMS,0.802726
6865,"""L'art pour l'espoir 3""",Exposition/Peinture/Sculpture/Photographie,**La Communauté Municipale de Santé est heureu...,5 - 30 janvier,Communauté Municipale de Santé - CMS,0.798172
5132,Peinture : Tu peins quoi ?,Atelier littéraire autour de la peinture,Mercredi 17 juin - 14h30 Atelier littéraire au...,"Jeudi 18 juin, 14h30",Médiathèque Empalot,0.796984
9207,"Jacques Mataly - ""L'horizon incertain""",Exposition photographique à la Galerie Ombres ...,L'exposition **Cette exposition intègre la pro...,28 mai - 31 juillet,Librairie Ombres blanches,0.796870



REQUÊTE : Je cherche un activité pour enfants


,title,description,longDescription,dateRange,location_name,score
11575,Les bébés explorateurs découvrent le monde,Atelier pour les 0-3 ans à la médiathèque jeun...,Atelier pour les tout-petits Une invitation à ...,"Samedi 25 juillet, 10h30",Muséum de Toulouse,0.776338
9757,Toulouse Plages - 18 août,Toulouse plages 2026 vous accueille à la prair...,### **Le programme de la journée** * 9h30-10h3...,"Mardi 18 août, 09h30",Prairie des Filtres - Toulouse Plages,0.772221
225,LES « SUPER-HEROS DES ARCHIVES » - Activités e...,Les Archives départementales seront ouvertes à...,### **Des activités sont prévues pour les enfa...,19 et 20 septembre,Archives départementales de la Haute-Garonne,0.771444
9464,Dalí s'invite à Toulouse,Une première : Toulouse accueille Dalí. Plus d...,### Des oeuvres rarement exposées en France L'...,5 décembre 2025 - 8 février 2026,Hôtel Albert 1er,0.769061
12172,Cherchez la petite bête !,Atelier de modelage,De nombreux animaux se dissimulent sur les pie...,"Mercredi 22 octobre 2025, 10h30","Musée Saint-Raymond, musée d'Archéologie de To...",0.768945



REQUÊTE : Je veux écouter du jazz


...

,title,description,longDescription,dateRange,location_name,score
3897,Visite au coeur d'une ressourcerie,Venez découvrir les coulisses de La Glanerie,"Pour la Semaine du Réemploi Solidaire, venez d...","Mercredi 1 octobre 2025, 14h00",La Glanerie,0.741846
8113,Nocturne étudiante :\nLa soirée insolite,L’insolite investit le musée pour quelques heu...,Du 6 au 15 mars les musées et monuments de la ...,"Jeudi 12 mars, 18h30",Musée des Arts Précieux Paul-Dupuy,0.738344
1297,Café-curiosité : drôles de bêtes !,Jeune public / Dès 8 ans,"Au Moyen Âge, dragons, sirènes et autres drôle...","Samedi 4 avril, 16h30",Micro-Folie Toulouse Ernest-Renan,0.737892
1130,Made in Asia,Festival des cultures de l'Asie d'aujourd'hui,19e édition du festival Made in Asia revient p...,11 - 19 avril,Multi-lieux,0.732960
13013,[Espace Roguet] ALLER SIMPLE – Compagnie HéTér...,"Prenant le parti de nous distraire, Aller Simp...",\ ### **ALLER SIMPLE** HéTéroKlite ‎‎‎‎ ‎ ‎ ‎ ...,"Vendredi 10 octobre 2025, 20h30",Espace Roguet,0.730278



REQUÊTE : Je voudrais une activité accessible aux personnes à mobilité réduite


,title,description,longDescription,dateRange,location_name,score
3448,Stand Cartoucirc - LGSV,"Fabrication de petits personnages ""têtes à gaz...",**Cartou Circ est une ressourcerie de quartier...,"Samedi 27 septembre 2025, 11h00",Les Halles de la Cartoucherie,0.817087
3456,La Grande Semaine Végétale X Les Halles de la ...,"🌱 LGSV, c’est une semaine festive et engagée, ...","🌱 LGSV, c’est une semaine festive et engagée à...","Samedi 27 septembre 2025, 11h00",Les Halles de la Cartoucherie,0.814732
5177,Temps fort braille : sensibilisation à la défi...,Des activités pour se sensibiliser de manière ...,**ÉVÉNEMENT ANNULÉ** Des activités pour se sen...,"Samedi 6 juin, 14h00",Médiathèque José Cabanis,0.810121
13363,HOLIDAY ON ICE,Grand Spectacle,HOLIDAY ON ICE - HORIZONS L'horizon est cette ...,6 - 8 mars,Zénith de Toulouse,0.809305
2473,SANTA,Variété / Chanson française,Santa a touché les cœurs avec son immense « Po...,"Dimanche 23 novembre 2025, 18h00",Zénith de Toulouse,0.808564



REQUÊTE : Je cherche un événement pour adolescents


,title,description,longDescription,dateRange,location_name,score
10781,Teen vidéo,Cinéma & images / Les Vidéophages / Courts-mét...,**Les Vidéophages** proposent des projections ...,"Mercredi 28 janvier, 15h00",Centre Culturel Espace Job,0.779135
10984,Teen vidéo,Cinéma & images / Les Vidéophages / Dès 12 ans,**Les Vidéophages** proposent des projections ...,"Mercredi 8 octobre 2025, 15h00",Centre Culturel Espace Job,0.772221
9331,Lâcher de caisses à savon,Un événement festif proposé par l'organisateur...,Dans le cadre du Carnaval Le C.O.C.U. (Comité ...,"Samedi 20 juin, 14h00",Avenue de la Colonne,0.766379
8496,Tempêtes adolescentes : visions croisées de la...,Jeudi 30 avril 2026 à 19h45. Dans le cadre du ...,NaN,"Jeudi 30 avril, 20h30",Le Cratère,0.761779
10146,Teen Vidéo,Projection de courts métrages / Les Vidéophage...,Les Vidéophages proposent des projections de c...,"Mercredi 30 septembre, 15h00",Centre Culturel Espace Job,0.758784
